# 9.2 不知道時怎麼回答？


盒內真有3球，助手沒看到，猜3即使撞中也沒有根據；若已明說3，卻還一律澄清，也沒有使用材料。只改「球數是否可見」，理想回答就應改變。

visible_count保存可見數量，沒有時None。作者另知hidden_count，但生成目標的規則只讀可見欄位，不能偷偷把隱值教成猜中的獎勵。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
examples = [
    {"visible_count": 3, "hidden_count": 3},
    {"visible_count": None, "hidden_count": 3},
]
for row in examples:
    visible = row["visible_count"]
    target = str(visible) if visible is not None else "看不到球數，請提供數量或圖片。"
    print("可見", visible, "→", target)

輸入兩筆的隱藏真值都為3，只有可見資訊不同。`visible`只取可見欄位，`is not None`檢查是否有值。這裡把分支縮成一行：`A if 條件 else B`表示條件成立就取A，不成立就取B。因此有值時`str`把整數轉成回答文字，無值時給出說明缺項的目標句。輸出第一行`可見 3 → 3`，第二行`可見 None → 看不到球數，請提供數量或圖片。`。關鍵是產生回答的那行從未讀取`hidden_count`。

這是人工規則產生訓練目標，不是模型已具備自我覺察的展示。真正訓練可以把「盒中有3顆球，問幾顆」與「一個未開的盒子，問幾顆」做成對話，讓模型學兩種回答情況。測試時再換球數與問法，查看它是否把缺資訊的題一律猜最常見數字，或把資訊完整的題也過度澄清。

缺球數就說清欠什麼；盒色紅通常不能推出數量，除非材料另給「每紅盒固定3球」規則。可回答與欠資訊兩組都要留出，分別查答對和適當澄清，避免碰巧猜中或每題說不知道過關。原問句和改問法的差別最後統一檢查。

練習替第二筆加`"box_color": "紅"`，保持產生目標的判斷不變。先預測第二行仍要求球數，然後重跑核對。若你想讓顏色影響答案，就先寫出並提供顏色到數量的規則；單純增加無關欄位不應讓猜測突然變成確定知識。

<details>
<summary>補充：實作約定與原始紀錄</summary>

人工規則也要用來檢查訓練後的模型。[9.8的混合版](https://birdhackor.github.io/tiny-perceptron-vlm/9.8.html)在原問句`count=?；有幾顆？`會回答`資訊不足，請提供數量。`；保持盒子與缺資訊條件不變，改問`盒子1；count=?；能確定球數嗎？`，卻回答`6`。這個對照提醒我們：學會在某句後面接澄清句，尚不等於能判斷不同問法中的資訊是否足夠。共同訓練設定與整組改問法結果會在9.8一起閱讀。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
